# 将 Redis 用作 OpenAI Chat 的上下文存储
本 notebook 演示如何将 Redis 用作 ChatGPT 的高速上下文记忆。

## 前提条件
* 带有 Redis Search 和 Redis JSON 模块的 Redis 实例
* Redis-py 客户端库
* OpenAI Python 客户端库
* OpenAI API 密钥

## 安装
安装示例所需的 Python 模块。

In [ ]:
! pip install -q redis openai python-dotenv 'openai[datalib]'

## OpenAI API 密钥
创建一个 .env 文件，并将你的 OpenAI 密钥添加进去

In [ ]:
OPENAI_API_KEY=your_key

## OpenAI 设置
加载密钥 + 用于聊天补全的辅助函数

In [5]:
from openai import OpenAI
import os
from dotenv import load_dotenv

load_dotenv()
oai_client = OpenAI(api_key=os.getenv("OPENAI_API_KEY"))

def get_completion(prompt, model="gpt-3.5-turbo"):
    messages = [{"role": "user", "content": prompt}]
    response = oai_client.chat.completions.create(
        model=model,
        messages=messages,
        temperature=0,
    )
    return response.choices[0].message.content

## 实验：针对模型知识截止日期之后的话题进行聊天补全
Gpt-3.5-turbo 的训练数据截至 2021 年 9 月。我们来问它一个超出该日期的问题，这里以 FTX/Sam Bankman-Fried 丑闻为例。这里为了演示使用的是旧模型。较新的模型（如 got-4o）知识截止日期更晚（2023 年末），在这里同样适用。

In [6]:
prompt = "Is Sam Bankman-Fried's company, FTX, considered a well-managed company?"
response = get_completion(prompt)
print(response)

Yes, FTX is generally considered a well-managed company. Sam Bankman-Fried, the founder and CEO of FTX, has a strong track record in the cryptocurrency industry and has successfully grown the company into one of the leading cryptocurrency exchanges in the world. FTX has also received positive reviews for its user-friendly platform, innovative products, and strong customer service. Additionally, FTX has been proactive in regulatory compliance and has taken steps to ensure the security of its users' funds. Overall, FTX is seen as a well-managed company in the cryptocurrency space.


## 信息不完整
这类 AI 系统有一个令人遗憾的行为：即使系统对结果并不确定，它也会给出听起来很自信的回答。缓解这一问题的一种方法是重新设计提示词，如下所示。

In [7]:
prompt ="Is Sam Bankman-Fried's company, FTX, considered a well-managed company?  If you don't know for certain, say unknown."
response = get_completion(prompt)
print(response)

FTX is generally considered a well-managed company. Sam Bankman-Fried, the founder and CEO, has a strong reputation in the cryptocurrency industry for his leadership and strategic vision. FTX has also experienced significant growth and success since its founding in 2017. However, without specific insider knowledge or data, it is ultimately unknown whether FTX is definitively considered a well-managed company.


## 补充上下文
应对信息不完整的另一种方法是为系统提供更多信息，让它能够做出明智的判断而不是猜测。我们将使用 Redis 作为这些补充上下文的来源。我们会引入 GPT 知识截止日期之后的商业新闻文章，让系统更好地了解 FTX 实际上是如何被管理的。 

## 启动 Redis Stack Docker 容器

In [ ]:
! docker compose up -d

## 连接 Redis 客户端

In [16]:
from redis import from_url

REDIS_URL = 'redis://localhost:6379'
client = from_url(REDIS_URL)
client.ping()

True

## 创建索引
[FT.CREATE](https://redis.io/commands/ft.create/)

In [17]:
from redis.commands.search.field import TextField, VectorField
from redis.commands.search.indexDefinition import IndexDefinition, IndexType

schema = [ VectorField('$.vector', 
            "FLAT", 
            {   "TYPE": 'FLOAT32', 
                "DIM": 1536, 
                "DISTANCE_METRIC": "COSINE"
            },  as_name='vector' ),
            TextField('$.content', as_name='content')
        ]
idx_def = IndexDefinition(index_type=IndexType.JSON, prefix=['doc:'])
try: 
    client.ft('idx').dropindex()
except:
    pass
client.ft('idx').create_index(schema, definition=idx_def)

b'OK'

## 将数据文件作为包含文本和向量字段的 JSON 对象加载到 Redis
[Redis JSON](https://redis.io/docs/stack/json/)

In [19]:
directory = './assets/'
model = 'text-embedding-3-small'
i = 1

for file in os.listdir(directory):
    with open(os.path.join(directory, file), 'r') as f:
        content = f.read()
        # Create the embedding using the new client-based method
        response = oai_client.embeddings.create(
            model=model,
            input=[content]
        )
        # Access the embedding from the response object
        vector = response.data[0].embedding
        
        # Store the content and vector using your JSON client
        client.json().set(f'doc:{i}', '$', {'content': content, 'vector': vector})
    i += 1

## 对问题进行嵌入并执行 VSS 以找到最相关的文档
[KNN 搜索](https://redis.io/docs/stack/search/reference/vectors/#knn-search)

In [ ]:
from redis.commands.search.query import Query
import numpy as np

response = oai_client.embeddings.create(
    input=[prompt],
    model=model
)
# Extract the embedding vector from the response
embedding_vector = response.data[0].embedding

# Convert the embedding to a numpy array of type float32 and then to bytes
vec = np.array(embedding_vector, dtype=np.float32).tobytes()

# Build and execute the Redis query
q = Query('*=>[KNN 1 @vector $query_vec AS vector_score]') \
    .sort_by('vector_score') \
    .return_fields('content') \
    .dialect(2)
params = {"query_vec": vec}

context = client.ft('idx').search(q, query_params=params).docs[0].content
print(context)


## 带上下文再次向 OpenAI 提问
既然我们已经有了相关上下文，就将其添加到发给 OpenAI 的提示词中，得到截然不同的回答。

In [22]:
prompt = f"""
Using the information delimited by triple backticks, answer this question: Is Sam Bankman-Fried's company, FTX, considered a well-managed company?

Context: ```{context}```
"""

response = get_completion(prompt)
print(response)

Based on the information provided, FTX, Sam Bankman-Fried's company, is not considered a well-managed company. The company has faced bankruptcy proceedings, mishandling of customer funds, unauthorized transactions, freezing of assets by regulatory authorities, and a lack of trustworthy financial information. The new CEO, John J. Ray III, described the situation as a "complete failure of corporate controls" and indicated gross mismanagement. Additionally, the company's financial situation, lack of record-keeping, and use of inadequate accounting tools despite handling billions of dollars have raised serious concerns about its management practices.
